# 03b — BAP validation support top-up

Questo notebook **non rigenera e non modifica** il dataset v1.1 da 29.240 transizioni. Usa come base l'Input Kaggle creato dall'output persistente precedente e genera un secondo shard piccolo per correggere l'unico blocker scientifico reale: in `validation` erano stati osservati 3 episodi positivi `backpropagating_ap`, contro il minimo preregistrato di 4. Se KaggleHub ha raccolto i molti file in `archive.zip`, la base viene estratta in `/kaggle/temp`, che non entra nell'output finale.

Il batch è deciso prima di osservare i risultati: 8 nuovi seed, 8 snapshot distinti, la ricetta BAP positiva già validata nel pilot e conservazione obbligatoria di tutti gli episodi. Non si abbassano soglie, non si trasformano fallimenti in hard-negative e non si scelgono a posteriori solo i seed riusciti. Il risultato finale resta una composizione logica di due shard separati. **NEURON deve partire in un processo Python nuovo:** per questo notebook usare `No persistence` o al massimo `Files only`, mai la persistenza delle variabili, e non rieseguire la preparazione del teacher nella stessa sessione.

## 1. Checkout riproducibile e teacher canonico

In [ ]:
import importlib, json, os, shutil, subprocess, sys
from pathlib import Path

ELM_REPOSITORY = "https://github.com/Zagred47/giada.git"
ELM_REF = os.environ.get("HAYFLOW_ELM_REF", "main")
TEACHER_REPOSITORY = "https://github.com/SelfishGene/neuron_as_deep_net.git"
TEACHER_COMMIT = "074c4666300a8ad246601dab179a97a6942f0f29"
NOTEBOOK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd().resolve()
WORKSPACE = NOTEBOOK_ROOT / "hayflow_workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)

def run(command, cwd=None):
    print("+", " ".join(map(str, command)), flush=True)
    subprocess.run(list(map(str, command)), cwd=cwd, check=True)

elm_override = os.environ.get("HAYFLOW_ELM_REPO")
mounted = [Path(elm_override).expanduser()] if elm_override else []
mounted.extend([Path.cwd(), *Path.cwd().parents])
ELM_REPO = next((p.resolve() for p in mounted if (p / "src" / "hayflow_teacher").is_dir()), None)
if ELM_REPO is None:
    ELM_REPO = WORKSPACE / "elmneuron"
    if not (ELM_REPO / ".git").is_dir():
        run(["git", "clone", ELM_REPOSITORY, ELM_REPO])
    run(["git", "fetch", "origin", ELM_REF], cwd=ELM_REPO)
    run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=ELM_REPO)
TEACHER_REPO = Path(os.environ.get("HAYFLOW_TEACHER_REPO", ELM_REPO.parent / "neuron_as_deep_net")).expanduser().resolve()
if not (TEACHER_REPO / ".git").is_dir():
    run(["git", "clone", TEACHER_REPOSITORY, TEACHER_REPO])
run(["git", "checkout", "--detach", TEACHER_COMMIT], cwd=TEACHER_REPO)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=TEACHER_REPO, text=True).strip() == TEACHER_COMMIT
print("Owned repository:", ELM_REPO)
print("Canonical teacher:", TEACHER_REPO)

## 2. Dipendenze e MOD originali

In [ ]:
run([sys.executable, "-m", "pip", "install", "--quiet", "neuron==8.2.7", "numpy", "pandas", "matplotlib", "h5py", "pyarrow", "pyyaml"])
SIMULATION_DIR = TEACHER_REPO / "L5PC_NEURON_simulation"
if not list(SIMULATION_DIR.rglob("libnrnmech.so")):
    nrnivmodl = shutil.which("nrnivmodl") or str(Path(sys.executable).parent / "nrnivmodl")
    run([nrnivmodl, "mods"], cwd=SIMULATION_DIR)
assert list(SIMULATION_DIR.rglob("libnrnmech.so")), "MOD compilation failed"
sys.path.insert(0, str(ELM_REPO))
for name in tuple(sys.modules):
    if name == "src.hayflow_teacher" or name.startswith("src.hayflow_teacher.") or name == "src.hayflow_data" or name.startswith("src.hayflow_data."):
        sys.modules.pop(name, None)
importlib.invalidate_caches()
print("Teacher mechanisms compiled without source changes.")

## 3. Individuazione dell'Input base e della calibrazione 01b

In [ ]:
import time, zipfile

def is_valid_base_root(path):
    path = Path(path)
    required = [path / "transition_dataset.h5", path / "validation_report.json", path / "artifact_index.json", path / "episodes.parquet"]
    if not all(item.is_file() for item in required):
        return False
    try:
        report = json.loads((path / "validation_report.json").read_text(encoding="utf-8"))
        return int(report.get("exhaustive_replay", {}).get("replayed_transition_count", -1)) == 29240
    except Exception:
        return False

def extract_zip_with_progress(archive, destination):
    archive, destination = Path(archive), Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as source:
        members = [row for row in source.infolist() if not row.is_dir()]
        total = sum(row.file_size for row in members)
        free = shutil.disk_usage(destination).free
        assert free >= total + 2 * 2**30, f"Spazio insufficiente: servono {(total + 2 * 2**30) / 2**30:.1f} GiB, disponibili {free / 2**30:.1f} GiB"
        completed, started, last = 0, time.perf_counter(), 0.0
        print(f"Estrazione base: {total / 2**30:.2f} GiB, {len(members)} file", flush=True)
        for member in members:
            target = (destination / member.filename).resolve()
            assert target.is_relative_to(destination.resolve()), f"Percorso ZIP non sicuro: {member.filename}"
            target.parent.mkdir(parents=True, exist_ok=True)
            with source.open(member) as reader, target.open("wb") as writer:
                while True:
                    block = reader.read(8 * 1024 * 1024)
                    if not block:
                        break
                    writer.write(block)
                    completed += len(block)
                    now = time.perf_counter()
                    if now - last >= 10 or completed == total:
                        rate = completed / max(now - started, 1e-9)
                        eta = (total - completed) / max(rate, 1e-9)
                        print(f"[HayFlow][estrazione] {100 * completed / max(total, 1):5.1f}% | {rate / 2**20:.1f} MiB/s | ETA {eta / 60:.1f} min", flush=True)
                        last = now

base_override = os.environ.get("HAYFLOW_BASE_DATASET")
base_candidates = [Path(base_override).expanduser()] if base_override else []
if Path("/kaggle/input").is_dir():
    base_candidates.extend(path.parent for path in Path("/kaggle/input").rglob("validation_report.json"))
BASE_DATASET = next((path.resolve() for path in base_candidates if is_valid_base_root(path)), None)
if BASE_DATASET is None and Path("/kaggle/input").is_dir():
    archive_candidates = []
    for archive in Path("/kaggle/input").rglob("*.zip"):
        try:
            with zipfile.ZipFile(archive) as source:
                names = set(source.namelist())
            if any(name.endswith("transition_dataset.h5") for name in names) and any(name.endswith("validation_report.json") for name in names):
                archive_candidates.append(archive.resolve())
        except zipfile.BadZipFile:
            pass
    assert len(archive_candidates) == 1, f"Atteso un solo archivio base HayFlow, trovati: {archive_candidates}"
    extracted = Path("/kaggle/temp/hayflow_targeted_base_v1_1")
    extracted_candidates = [path.parent for path in extracted.rglob("validation_report.json")] if extracted.exists() else []
    BASE_DATASET = next((path.resolve() for path in extracted_candidates if is_valid_base_root(path)), None)
    if BASE_DATASET is None:
        if extracted.exists() and any(extracted.iterdir()):
            raise RuntimeError(f"Estrazione parziale trovata in {extracted}; riavvia la sessione prima di riprovare.")
        extract_zip_with_progress(archive_candidates[0], extracted)
        extracted_candidates = [path.parent for path in extracted.rglob("validation_report.json")]
        BASE_DATASET = next((path.resolve() for path in extracted_candidates if is_valid_base_root(path)), None)
assert BASE_DATASET is not None, "Dataset base HayFlow non trovato né come cartella né dentro l'archivio Kaggle."

calibration_override = os.environ.get("HAYFLOW_CALIBRATION_SOURCE")
calibration_candidates = [Path(calibration_override).expanduser()] if calibration_override else []
calibration_candidates.append(Path("/kaggle/input/datasets/alessandrobelli/hayflow-dendritic-protocol-calibration/hayflow_dendritic_protocol_calibration"))
if Path("/kaggle/input").is_dir():
    calibration_candidates.extend(path.parent for path in Path("/kaggle/input").rglob("selected_dendritic_protocols.json"))
CALIBRATION_SOURCE = next((path.resolve() for path in calibration_candidates if path.exists()), None)
assert CALIBRATION_SOURCE is not None, "Calibrazione 01b non trovata. Imposta HAYFLOW_CALIBRATION_SOURCE."
print("Base read-only:", BASE_DATASET)
print("Calibration source:", CALIBRATION_SOURCE)

## 4. Teacher, contratto e verifica della base

Il calcolo SHA-256 del file base da circa 6 GiB può richiedere alcuni minuti e stampa percentuale, velocità ed ETA. È intenzionale: lega il nuovo manifest esattamente allo shard che ha superato il replay 29.240/29.240.

In [ ]:
import yaml
from IPython.display import display
from src.hayflow_teacher import BapValidationSupportTopupSession, expected_audit_hashes

BASE_CONFIG_PATH = ELM_REPO / "configs" / "hayflow" / "transition_dataset_diagnostic.yml"
TARGET_CONFIG_PATH = ELM_REPO / "configs" / "hayflow" / "targeted_transition_dataset_v1_1.yml"
base_config = yaml.safe_load(BASE_CONFIG_PATH.read_text(encoding="utf-8"))
target_config = yaml.safe_load(TARGET_CONFIG_PATH.read_text(encoding="utf-8"))
OUTPUT_DIR = Path(os.environ.get("HAYFLOW_OUTPUT_DIR", NOTEBOOK_ROOT / "artifacts" / "bap_validation_support_topup_v3")).expanduser().resolve()
if OUTPUT_DIR.exists():
    raise RuntimeError(f"Output protetto già presente: {OUTPUT_DIR}. Per un nuovo tentativo usa una nuova HAYFLOW_OUTPUT_DIR.")
session = BapValidationSupportTopupSession(
    ELM_REPO, TEACHER_REPO,
    base_dataset=BASE_DATASET,
    calibration_source=CALIBRATION_SOURCE,
    dataset_config_path=TARGET_CONFIG_PATH,
    output_dir=OUTPUT_DIR,
    seed=base_config["runtime"]["seed"],
    expected_teacher_hashes=expected_audit_hashes(),
    native_snapshot_stride=target_config["storage"]["native_snapshot_stride_ms"],
)
teacher_report = session.prepare_teacher()
contract_report = session.prepare_targeted_contract()
base_report = session.verify_base_dataset(verify_large_hdf=True)
equilibrium_import = session.import_base_equilibrium()
display({"teacher": teacher_report, "contract": contract_report, "base": base_report, "equilibrium": equilibrium_import})
assert teacher_report["segment_count"] == 642 and base_report["valid"]

## 5. Preregistrazione e nuovi snapshot

Da qui il batch è congelato su disco prima della simulazione. Gli otto snapshot sono nuovi e appartengono soltanto alla validation.

In [ ]:
protocols, topup_plan = session.build_topup_plan()
snapshot_report = session.prepare_snapshot_bank(protocols, conditioning_ms=4)
display({"plan": topup_plan, "snapshots": snapshot_report})
assert topup_plan["episode_count"] == 8
assert topup_plan["transition_count"] == 640
assert snapshot_report["valid"] and snapshot_report["snapshot_count"] == 8

## 6. Generazione completa del piccolo shard BAP

In [ ]:
topup_manifest = session.generate_topup(protocols)
episodes = session._normalized_episodes(session.pd.read_parquet(OUTPUT_DIR / "episodes.parquet"))
bap_positive_count = sum("backpropagating_ap" in row["event_labels"] for row in episodes)
display({"manifest": topup_manifest, "retained_episode_count": len(episodes), "observed_bap_positive_count": bap_positive_count})
assert len(episodes) == 8, "Tutti gli episodi preregistrati devono essere conservati."

## 7. Replay esaustivo dello shard e validazione composita

Vengono riprodotte soltanto le 640 nuove transizioni. La prova del replay base viene verificata e referenziata, non ricalcolata. Se nessuno degli otto episodi produce un BAP valido, il notebook si ferma senza scartare casi o cambiare le soglie.

In [ ]:
validation_report = session.validate_topup_and_composite(protocols)
display(validation_report)
assert validation_report["valid"]
assert validation_report["topup"]["exhaustive_replay"]["replayed_transition_count"] == 640
assert validation_report["composite_support"]["topup_bap_positive_count"] >= 1

## 8. Download dello shard supplementare

Lo ZIP contiene soltanto il nuovo shard e il manifest composito; non duplica i 6 GiB della base Quick Saved. La cella usa il metodo Base64 → JavaScript → Blob che ha già funzionato in questo progetto.

In [ ]:
import base64
from IPython.display import Javascript, display

archive_base = NOTEBOOK_ROOT / "hayflow_bap_validation_support_topup_v3"
zip_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name))
encoded = base64.b64encode(zip_path.read_bytes()).decode("ascii")
display(Javascript(f"""
const b64 = '{encoded}';
const binary = atob(b64);
const bytes = new Uint8Array(binary.length);
for (let i = 0; i < binary.length; i++) bytes[i] = binary.charCodeAt(i);
const blob = new Blob([bytes], {{type: 'application/zip'}});
const url = URL.createObjectURL(blob);
const anchor = document.createElement('a');
anchor.href = url;
anchor.download = '{zip_path.name}';
document.body.appendChild(anchor);
anchor.click();
anchor.remove();
setTimeout(() => URL.revokeObjectURL(url), 60000);
"""))
print(f"Download avviato: {zip_path.name} ({zip_path.stat().st_size / 2**20:.1f} MiB)")